<a href="https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents a single daily performance report for a unique combination of client_id, content_id, and date

One row represents: A single daily performance report for a unique combination of client_id, content_id, and date.

Time Window: Training and feature engineering are restricted to the mid-panel month (2026-03-01 to 2026-03-31). The final month (2026-06 / _sample) is strictly reserved as a sealed test set.

### Setting `HF_TOKEN` using Colab Secrets

1.  **Open the Secrets tab:** In the left-hand panel of your Colab notebook, look for the '🔑' (Secrets) icon and click on it.
2.  **Add a new secret:** Click on 'Add new secret'.
3.  **Name the secret:** For the 'Name' field, type `HF_TOKEN` (it's crucial that the name matches what your code expects).
4.  **Enter your token:** In the 'Value' field, paste your actual Hugging Face token.
5.  **Enable notebook access:** Make sure the 'Notebook access' toggle is switched on for this secret.

Once you've done this, you can access the `HF_TOKEN` in your Python code like this:

After setting the `HF_TOKEN` in Colab Secrets and running the cell above to load it, you should then **re-run cell `bP59hNOzVb-2`** so that the `con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}')")` line picks up the newly available token. This will allow `duckdb` to authenticate and access the Hugging Face dataset.

In [17]:
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 1. Authenticate DuckDB with Hugging Face Token from Colab Secrets / Environment
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.getenv("HF_TOKEN", "")

con = duckdb.connect()
if hf_token:
    con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}')")

# Base paths
HF_DATASET_URL = "hf://datasets/FlyRank/internship-warehouse"
FACT_TABLE = f"{HF_DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet"
CLIENT_TABLE = f"{HF_DATASET_URL}/dim_clients.parquet" # Corrected to single file dim_clients.parquet

print("DuckDB initialized and connected to FlyRank warehouse.")

DuckDB initialized and connected to FlyRank warehouse.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Field Bucket ClassificationFeatures (Inputs knowable at time $t$):feat_clicks_7d_avg: Trailing 7-day average GSC clicks ($t-7$ to $t-1$).feat_impressions_7d_sum: Trailing 7-day sum of GSC impressions ($t-7$ to $t-1$).feat_ctr_7d_avg: Trailing 7-day average CTR ($t-7$ to $t-1$).feat_position_7d_avg: Trailing 7-day average search position ($t-7$ to $t-1$).feat_days_since_client_start: Elapsed days between client onboarding (gsc_data_start) and report date $t$.Label (Target to Predict):target_next_day_clicks: Organic clicks observed on the following day ($t+1$).Context (Identifiers & Joining Keys):client_id: Unique identifier for the client entity.content_id: Unique identifier for the specific content URL/page.date: Observation date timestamp.Excluded Fields (With Reasoning):gsc_clicks (same-day $t$ raw metric): Excluded from features to prevent same-day target contamination.ga4_conversions / ga4_revenue: Excluded due to inconsistent GA4 tracking setup across client tiers during early historical windows.raw_query_terms: Excluded due to high dimensionality and Google-anonymized search query missingness.

In [7]:

schema_df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{FACT_TABLE}') LIMIT 1;").df()
print("Fact Table Column Types:")
print(schema_df[['column_name', 'column_type']])

Fact Table Column Types:
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt    

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain Uniqueness: Confirms (client_id, content_id, date) forms a unique primary key with 0 duplicate collisions.

Slice Span & Volume: Confirms row count and strict date boundary alignment inside month=2026-03.

Data Availability (IS TRUE): Uses explicit IS TRUE boolean filtering to quantify active tracking windows.

In [12]:
# Updated table paths for FlyRank warehouse
CLIENT_TABLE_FILE = f"{HF_DATASET_URL}/dim_clients.parquet"

# Fallback path if stored inside a directory
CLIENT_TABLE_GLOB = f"{HF_DATASET_URL}/dim_clients/*.parquet"

# Query 3: Fixed Availability Verification with IS TRUE
q3_avail = f"""
SELECT
    COUNT(f.*) as total_fact_rows,
    COUNT(CASE WHEN (c.gsc_data_start <= f.report_date) IS TRUE THEN 1 END) as gsc_active_rows,
    ROUND(100.0 * COUNT(CASE WHEN (c.gsc_data_start <= f.report_date) IS TRUE THEN 1 END) / COUNT(*), 2) as availability_pct
FROM read_parquet('{FACT_TABLE}') f
LEFT JOIN read_parquet('{CLIENT_TABLE_FILE}') c ON f.client_hash_id = c.client_hash_id;
"""

print("--- Query 3: Availability Verification (IS TRUE) ---")
try:
    print(con.sql(q3_avail).df())
except Exception as e:
    # If dim_clients.parquet is in a folder rather than root file:
    q3_avail_alt = q3_avail.replace(CLIENT_TABLE_FILE, CLIENT_TABLE_GLOB)
    print(con.sql(q3_avail_alt).df())

--- Query 3: Availability Verification (IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_fact_rows  gsc_active_rows  availability_pct
0          9841378          9617697             97.73


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

feat_clicks_7d_avg: Knowable at decision moment $t$ because it aggregates historical clicks up to $t-1$.feat_impressions_7d_sum: Knowable at decision moment $t$ because it sums logged search impressions up to $t-1$.feat_ctr_7d_avg: Knowable at decision moment $t$ because it averages prior Search Console CTR up to $t-1$.feat_position_7d_avg: Knowable at decision moment $t$ because rank positions up to $t-1$ are fully ingested.feat_days_since_client_start: Knowable at decision moment $t$ because client onboarding timestamp is static metadata.Named Data Limitation-Unbalanced History Depth & Early Integration Lag:The panel exhibits unbalanced historical depth where client properties onboarded at different times (gsc_data_start). Early dates contain lower density for newer domains, and search metrics represent directional aggregate trends rather than raw searcher-level identity.

In [18]:
# 1. Feature Extraction SQL (Using exact table path & column names)
# CLIENT_TABLE_FILE = f"{HF_DATASET_URL}/dim_clients.parquet" # Removed, using global CLIENT_TABLE

feature_sql = f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.report_date,

    -- Feature 1: 7d Clicks Avg (Historical t-7 to t-1)
    AVG(f.gsc_clicks) OVER (
        PARTITION BY f.client_hash_id, f.content_hash_id
        ORDER BY f.report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as feat_clicks_7d_avg,

    -- Feature 2: 7d Impressions Sum (Historical t-7 to t-1)
    SUM(f.gsc_impressions) OVER (
        PARTITION BY f.client_hash_id, f.content_hash_id
        ORDER BY f.report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as feat_impressions_7d_sum,

    -- Feature 3: 7d CTR Avg (Historical t-7 to t-1)
    AVG(f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions, 0)) OVER (
        PARTITION BY f.client_hash_id, f.content_hash_id
        ORDER BY f.report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as feat_ctr_7d_avg,

    -- Feature 4: 7d Position Avg (Historical t-7 to t-1)
    AVG(f.gsc_avg_position) OVER (
        PARTITION BY f.client_hash_id, f.content_hash_id
        ORDER BY f.report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as feat_position_7d_avg,

    -- Feature 5: Days since client onboarding
    (f.report_date - c.gsc_data_start) as feat_days_since_client_start,

    -- Honest Target (t+1 clicks)
    LEAD(f.gsc_clicks, 1) OVER (
        PARTITION BY f.client_hash_id, f.content_hash_id
        ORDER BY f.report_date
    ) as target_next_day_clicks

FROM read_parquet('{FACT_TABLE}') f
LEFT JOIN read_parquet('{CLIENT_TABLE}') c
    ON f.client_hash_id = c.client_hash_id
"""

# Extract features and drop boundary NaNs (from rolling window and LEAD)
df_features = con.sql(feature_sql).df().dropna()
print(f"Feature dataset built: {len(df_features):,} rows.")

# 2. Springing the Trap: Injecting Target-Derived Leakage Column
df_features['LEAKED_TRAP_COLUMN'] = df_features['target_next_day_clicks'] * 0.99 + np.random.normal(0, 0.01, len(df_features))

features_leaked = [
    'feat_clicks_7d_avg', 'feat_impressions_7d_sum', 'feat_ctr_7d_avg',
    'feat_position_7d_avg', 'feat_days_since_client_start', 'LEAKED_TRAP_COLUMN'
]

X_leak = df_features[features_leaked]
y = df_features['target_next_day_clicks']

model_leak = RandomForestRegressor(n_estimators=10, random_state=42)
model_leak.fit(X_leak, y)
r2_leaked = r2_score(y, model_leak.predict(X_leak))
print(f"TRAP ACTIVE — Artificial R² Score (Leaked Feature): {r2_leaked:.4f}")

# 3. Removing the Trap: Honest Model Evaluation
honest_features = [
    'feat_clicks_7d_avg', 'feat_impressions_7d_sum', 'feat_ctr_7d_avg',
    'feat_position_7d_avg', 'feat_days_since_client_start'
]

df_honest = df_features.drop(columns=['LEAKED_TRAP_COLUMN'])
X_honest = df_honest[honest_features]

model_honest = RandomForestRegressor(n_estimators=10, random_state=42)
model_honest.fit(X_honest, y)
r2_honest = r2_score(y, model_honest.predict(X_honest))
print(f"HONEST MODEL — Realistic R² Score (No Leakage): {r2_honest:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature dataset built: 4,034,510 rows.
TRAP ACTIVE — Artificial R² Score (Leaked Feature): 1.0000
HONEST MODEL — Realistic R² Score (No Leakage): 0.9276


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.